# Step 07.3 — Build the `(B,T,C)` Representation

到这里，我们已经分别学过：

```text
B = batch axis
T = token / sequence axis
C = embedding / feature axis
```

现在把它们真正合成 Transformer 的标准 hidden-state 张量：

```text
(B,T) token IDs
   ↓ nn.Embedding(V,C)
(B,T,C) hidden representation
```

这一节完整解决 5 个问题：

1. `(B,T,C)` 是怎样从 batch token IDs 产生的；
2. 三个 axis 分别代表什么，以及常见切片怎样读；
3. 为什么 `C=n_embd` 会成为整个 Transformer 的 model width；
4. 哪些操作在混合 C，哪些操作以后会混合 T；
5. `(B,T,C)` 的内存和可视化直觉，以及为什么 token embedding 本身还没有 position information。

## 1. 从 `(B,T)` token IDs 到 `(B,T,C)`：完整数据接口

先构造一个极小 batch：

```text
B = 2
T = 4
V = 6
C = 5
```

token IDs：

```text
X =
[[2, 0, 2, 5],
 [1, 4, 3, 1]]

X.shape = (2,4)
```

这里每个格子仍然只是一个离散类别 ID。

经过 `nn.Embedding(V,C)` 后，每个标量 ID 被替换成一个长度为 C 的 row vector，因此：

```text
(B,T)
  ↓
(B,T,C)
```

In [ ]:
import mlx.core as mx
import mlx.nn as nn

B, T, V, C = 2, 4, 6, 5

X = mx.array([
    [2, 0, 2, 5],
    [1, 4, 3, 1],
])

wte = nn.Embedding(V, C)
h = wte(X)

mx.eval(h)

print("X shape        :", X.shape)
print("wte.weight     :", wte.weight.shape)
print("h shape        :", h.shape)


这里可以逐格理解：

```text
X[b,t]           -> 一个整数 token ID
h[b,t,:]         -> 这个 token 的 C 维向量
```

例如：

```text
X[0,0] = 2
h[0,0,:] = wte.weight[2]
```

因此 `(B,T,C)` 不是抽象约定，而是把 `B*T` 个 token ID 每一个都替换成 C 维 feature vector 后自然得到的 shape。

In [ ]:
token_id = int(X[0, 0].item())
from_batch = h[0, 0, :]
from_table = wte.weight[token_id]

mx.eval(from_batch, from_table)

print("token ID:", token_id)
print("h[0,0,:] shape:", from_batch.shape)
print("same as weight[token_id]:", bool(mx.allclose(from_batch, from_table)))

## 2. 三个 axis 到底分别是什么？——学会读 `(B,T,C)` 的切片

对：

```text
h.shape = (B,T,C)
```

三个 axis 的意义是：

```text
axis 0 = B = 哪一个 batch sample
axis 1 = T = 这个 sample 里的哪个 token position
axis 2 = C = 这个 token 当前有哪些 hidden features
```

因此常用切片可以直接翻译成人话。

In [ ]:
print("h[0, 2, :]  -> one token vector       :", h[0, 2, :].shape)
print("h[0, :, :]   -> one whole sequence     :", h[0, :, :].shape)
print("h[:, 2, :]   -> same position in batch :", h[:, 2, :].shape)
print("h[:, :, 1]   -> one feature everywhere :", h[:, :, 1].shape)

逐个读：

```text
h[b,t,:] -> (C,)
```

表示第 b 条 sequence 的第 t 个 token 的完整 feature vector。

```text
h[b,:,:] -> (T,C)
```

表示一整条 sequence 的所有 token vectors。

```text
h[:,t,:] -> (B,C)
```

表示 batch 中所有样本在同一个相对 position t 的表示。

```text
h[:,:,c] -> (B,T)
```

表示第 c 个 feature coordinate 在所有 batch / token position 上的数值。

注意：一个 C 维坐标通常不能简单解释成“这个维度就是名词”“这个维度就是情感”。它只是 learned representation 的一个坐标轴。

## 3. 为什么 `C=n_embd` 会成为整个 Transformer 的 Model Width？

Embedding 是模型第一次把 token 变成 C 维连续向量，但这个 C 不会在下一行代码就消失。

Transformer Block 通常把 `(B,T,C)` 当成统一的接口：

```text
(B,T,C)
   ↓ Block 1
(B,T,C)
   ↓ Block 2
(B,T,C)
   ↓ Block 3
(B,T,C)
```

因此 C 是整条 hidden-state highway 的宽度，常写成：

```text
C = n_embd = hidden size = model width
```

这些词在不同代码库中命名不同，但在我们的 MiniGPT 中指同一个核心维度。

### 为什么很多子层最后必须回到 C？

一个原因是 residual connection 以后要做：

```text
x + F(x)
```

如果：

```text
x.shape    = (B,T,C)
F(x).shape = (B,T,C)
```

才能逐元素相加。

所以很多模块内部可以暂时扩张，例如 MLP：

```text
C -> 4C -> C
```

但模块出口通常回到 C。Attention 也会先做 Q/K/V projection，最终再返回 `(B,T,C)`。

In [ ]:
same_width = nn.Linear(C, C, bias=False)
expanded = nn.Linear(C, 4 * C, bias=False)

h_same = same_width(h)
h_wide = expanded(h)

print("input               :", h.shape)
print("Linear(C,C)         :", h_same.shape)
print("Linear(C,4C)        :", h_wide.shape)
print("residual-compatible :", h.shape == h_same.shape)

这里再次看到 MLX `Linear` 的一个重要规则：

> 它作用在最后一个 feature axis 上，前面的 `(B,T)` axes 保留下来。

因此：

```text
(B,T,C) --Linear(C,Cout)--> (B,T,Cout)
```

这也是为什么 C 被称为 channel / hidden dimension：后面大量 feature transformation 都围绕最后这一维发生。

## 4. `C` 上的混合和 `T` 上的混合不是一回事

这一点是理解 Transformer 的核心分工。

对 `(B,T,C)`：

```text
C axis -> 一个 token 内部有哪些 feature
T axis -> sequence 中有哪些 token positions
```

`nn.Linear(C,C)` 或 MLP 对每个 `(b,t)` 位置应用同一套 weight，它会混合 C 中的 features，但不会让 position 0 直接读取 position 1 的 vector。

所以 MLP 的基本角色是：

```text
within-position feature transformation
```

而以后 Attention 的核心角色是：

```text
cross-position information mixing
```

In [ ]:
# 验证一个 C->C Linear 不会把不同 T 位置混在一起
proj = nn.Linear(C, C, bias=False)

base = h[0:1]  # shape (1,T,C)
changed = mx.array(base)

# 只改变 position 2 的全部 features
changed = mx.concatenate([
    changed[:, :2, :],
    changed[:, 2:3, :] + 10.0,
    changed[:, 3:, :],
], axis=1)

out_base = proj(base)
out_changed = proj(changed)

mx.eval(out_base, out_changed)

for t in range(T):
    same = bool(mx.allclose(out_base[:, t, :], out_changed[:, t, :]))
    print(f"position {t}: output unchanged? {same}")

你应该看到只有被修改的 position 2 的输出变化，其他 position 不变。

这就是“只沿 C 做 feature mixing”的直观证据。

等到 Attention，我们会第一次让一个 position 的输出依赖其他 position 的内容。

## 5. 把 `(B,T,C)` 看成一叠 `T×C` Feature Maps，并看到 Position 问题

对于一个固定 batch sample：

```text
h[b].shape = (T,C)
```

可以把它画成 heatmap：

```text
row    = token position t
column = feature dimension c
cell   = h[b,t,c]
```

这不是图像卷积里的 feature map，只是一个很方便的二维可视化视角。

In [ ]:
import matplotlib.pyplot as plt

sample_matrix = h[0].tolist()

fig, ax = plt.subplots(figsize=(7, 4))
image = ax.imshow(sample_matrix, aspect="auto")
ax.set_xlabel("feature / channel c")
ax.set_ylabel("token position t")
ax.set_title("One sequence represented as a T x C matrix")
ax.set_yticks(range(T))
fig.colorbar(image, ax=ax, label="hidden value")
plt.show()

### 内存为什么会随着 B、T、C 一起增长？

`(B,T,C)` 一共有：

```text
B*T*C
```

个数值。

如果是 fp32，每个数约 4 bytes，因此单个 hidden tensor 的粗略大小：

```text
B*T*C*4 bytes
```

实际训练还要保存很多层的中间 activation、gradient、optimizer state，所以总内存远不止一个 tensor。

In [ ]:
B_real = 32
T_real = 64
C_real = 128

values = B_real * T_real * C_real
mib_fp32 = values * 4 / 1024**2

print("values in one (B,T,C) tensor:", values)
print(f"fp32 size of one such tensor   : {mib_fp32:.2f} MiB")

### 但现在还缺一件关键东西：Position Information

Token embedding lookup 只由 token ID 决定。

例如序列：

```text
[2,0,5]
```

如果改成：

```text
[5,0,2]
```

embedding lookup 只是把同样的三行 vector 换了顺序；每一行 vector 本身并不知道“我是第 0 位”还是“我是第 2 位”。

数组 axis 当然保留了顺序，但 token representation 里还没有显式位置编码。

如果后续网络的计算对位置排列是 permutation-equivariant、又没有任何 positional signal，那么模型无法从 token content 本身知道绝对/相对顺序。

这就是下一节 7.4 的核心问题。

In [ ]:
seq_a = mx.array([2, 0, 5])
seq_b = mx.array([5, 0, 2])

emb_a = wte(seq_a)
emb_b = wte(seq_b)

mx.eval(emb_a, emb_b)

print("emb_a[0] equals token-2 row:", bool(mx.allclose(emb_a[0], wte.weight[2])))
print("emb_b[2] equals token-2 row:", bool(mx.allclose(emb_b[2], wte.weight[2])))
print("same token row moved to a different T position")

### 7.3 最终结论

- `(B,T)` 中每个整数 token ID 经 Embedding 变成 C 维向量，因此自然得到 `(B,T,C)`；
- `B` 选 batch sample，`T` 选 token position，`C` 选该 token 的 hidden feature coordinate；
- `h[b,t,:]` 是一个 token 的完整 C 维 hidden vector；
- `C=n_embd` 会成为 Transformer Block 之间共享的 model width / hidden size 接口；
- Linear/MLP 通常沿最后的 C 轴做 feature mixing，而保持 B/T axes；
- Attention 以后负责跨 T positions 进行信息混合；
- Residual connection 要求子层最终回到相同 `(B,T,C)` shape；
- `(B,T,C)` 的存储量与 B、T、C 的乘积成正比，但真实训练内存还包括大量中间状态；
- token embedding 本身只编码 token identity，不显式编码 position；
- 下一节 7.4：为什么序列模型必须表示顺序，RNN 为什么天然有顺序，而 Self-Attention 为什么需要额外 Position Information。